In [1]:
# Debug Notebook Cell 1: Setup
import sys
import os
os.chdir('..')  # Move to project root if in notebooks/
sys.path.append('.')

from qanda_module.config import setup_system
from qanda_module.ui_gradio import format_selection_display

# Setup system
helpers, qa_chain, config = setup_system("config.yaml")
print("✅ System loaded")

Loaded 475 panelist profile URLs
Total panellist responses: 17815
Panellist responses with missing speaker_id: 0
Created panelist_lookup with 475 entries for UI display
✅ System loaded


In [2]:
# In a notebook cell:
print("=== DEBUG PANELIST LOOKUP ===")
print(f"Total entries: {len(helpers.panelist_lookup)}")

# Check for Chris Bowen variations
bowen_matches = []
for key, value in helpers.panelist_lookup.items():
    if "chris" in key.lower() and "bowen" in key.lower():
        bowen_matches.append((key, value))

print(f"Chris Bowen matches: {bowen_matches}")

# Test the exact lookup logic
test_panelist = "CHRIS BOWEN"
matching_key = None
for key in helpers.panelist_lookup.keys():
    if key.upper() == test_panelist.upper():
        matching_key = key
        break

print(f"Looking for: '{test_panelist}'")
print(f"Found match: {matching_key}")
if matching_key:
    print(f"Data: {helpers.panelist_lookup[matching_key]}")

=== DEBUG PANELIST LOOKUP ===
Total entries: 475
Chris Bowen matches: [('Chris Bowen', ('Shadow Minister for Health', 'https://www.abc.net.au/qanda/chris-bowen/11517558'))]
Looking for: 'CHRIS BOWEN'
Found match: Chris Bowen
Data: ('Shadow Minister for Health', 'https://www.abc.net.au/qanda/chris-bowen/11517558')


In [2]:
# Debug Notebook Cell 2: Test format_selection_display directly
test_panelist = "CHRIS BOWEN"

# This should be the temp_lookup created in the UI
test_lookup = {
    "CHRIS BOWEN": ("Shadow Minister for Health", "https://www.abc.net.au/qanda/chris-bowen/11517558", "2020-04-06")
}

print(f"🔍 Testing with:")
print(f"  panelist: '{test_panelist}'")
print(f"  lookup data: {test_lookup[test_panelist]}")

# Test the function
result = format_selection_display(
    panelist=test_panelist,
    topic="",
    panelist_lookup=test_lookup
)

print(f"\n📝 Result:")
print(result)

🔍 Testing with:
  panelist: 'CHRIS BOWEN'
  lookup data: ('Shadow Minister for Health', 'https://www.abc.net.au/qanda/chris-bowen/11517558', '2020-04-06')

📝 Result:
**Current Focus:** 👤 [CHRIS BOWEN](https://www.abc.net.au/qanda/chris-bowen/11517558) *(Shadow Minister for Health, 2020-04-06)*


In [3]:
# Debug Notebook Cell 3: Test the actual helpers.panelist_lookup
print("🔍 Testing with actual helpers.panelist_lookup:")
print(f"Chris Bowen data: {helpers.panelist_lookup.get('Chris Bowen', 'NOT FOUND')}")

# Test with actual lookup
result2 = format_selection_display(
    panelist="CHRIS BOWEN",
    topic="", 
    panelist_lookup=helpers.panelist_lookup
)

print(f"\n📝 Result with actual lookup:")
print(result2)

🔍 Testing with actual helpers.panelist_lookup:
Chris Bowen data: ('Shadow Minister for Health', 'https://www.abc.net.au/qanda/chris-bowen/11517558')

📝 Result with actual lookup:
**Current Focus:** 👤 [Chris Bowen](https://www.abc.net.au/qanda/chris-bowen/11517558) *(Shadow Minister for Health)*


In [4]:
# Debug Notebook Cell 4: Test the actual UI flow
from qanda_module.data_processing import extract_panelist_name

# Simulate the UI flow
panelist_display = "CHRIS BOWEN (8 eps)"  # What comes from dropdown
clean_panelist = extract_panelist_name(panelist_display)

print(f"🔍 UI Flow Test:")
print(f"  dropdown value: '{panelist_display}'")
print(f"  clean_panelist: '{clean_panelist}'")

# Simulate the temp_lookup creation (from your fixed code)
matching_key = None
for key in helpers.panelist_lookup.keys():
    if key.upper() == clean_panelist.upper():
        matching_key = key
        break

print(f"  matching_key: '{matching_key}'")

if matching_key:
    profession, url = helpers.panelist_lookup[matching_key]
    temp_lookup = {clean_panelist: (profession, url, "2020-04-06")}
    print(f"  temp_lookup: {temp_lookup}")
    
    # Test final display
    result = format_selection_display(clean_panelist, "", temp_lookup)
    print(f"  final result: {result}")

🔍 UI Flow Test:
  dropdown value: 'CHRIS BOWEN (8 eps)'
  clean_panelist: 'CHRIS BOWEN'
  matching_key: 'Chris Bowen'
  temp_lookup: {'CHRIS BOWEN': ('Shadow Minister for Health', 'https://www.abc.net.au/qanda/chris-bowen/11517558', '2020-04-06')}
  final result: **Current Focus:** 👤 [CHRIS BOWEN](https://www.abc.net.au/qanda/chris-bowen/11517558) *(Shadow Minister for Health, 2020-04-06)*


In [3]:
# Debug Notebook Cell 1.5: Create the missing lookup
import duckdb
from qanda_module.data_processing import get_latest_panelist_data

# Connect to database and create lookup
con = duckdb.connect(config.duck_db_name)
panelist_lookup = get_latest_panelist_data(con)

# Attach to helpers object
helpers.panelist_lookup = panelist_lookup
helpers.con = con

print(f"✅ Created panelist_lookup with {len(panelist_lookup)} panelists")

Loaded 475 panelists using date-based latest episode logic
✅ Created panelist_lookup with 478 panelists


In [4]:
# Debug Notebook Cell 2: Check lookup data
print("🔍 Panelist Lookup Debug")
print(f"Total panelists in lookup: {len(helpers.panelist_lookup)}")

# Check for Chris Bowen specifically
target_name = "CHRIS BOWEN"
print(f"\nLooking for: '{target_name}'")

# Check exact match
if target_name in helpers.panelist_lookup:
    print(f"✅ Exact match found: {helpers.panelist_lookup[target_name]}")
else:
    print("❌ No exact match")

# Check case-insensitive matches
matches = []
for key, value in helpers.panelist_lookup.items():
    if "chris" in key.lower() and "bowen" in key.lower():
        matches.append((key, value))

print(f"\nCase-insensitive matches for Chris Bowen:")
for key, value in matches:
    print(f"  '{key}' → {value}")

🔍 Panelist Lookup Debug
Total panelists in lookup: 478

Looking for: 'CHRIS BOWEN'
❌ No exact match

Case-insensitive matches for Chris Bowen:
  'Chris Bowen' → ('', 'https://www.abc.net.au/qanda/chris-bowen/10641206')


: 